# 02 – Model training and export for the coach app

Trains the model behind `app/` on the cleaned dataset from notebook 01 and exports it as a small JSON model
card that the browser app evaluates itself (no server).

**Model.** Logistic regression on ratio features that is **monotone in the planned weight**: for a fixed
situation, a heavier 3rd attempt can never get a lower failure risk. Without that constraint the model learns
the selection effect seen in notebook 01 (coaches pick small jumps after a slow 2nd attempt, so small jumps
*look* risky) and would tell coaches that adding weight makes the lift safer. The constraint is what makes
"the heaviest weight within my risk tolerance" a meaningful question.

**Data use** (spec 06): fit on `train`; choose the threshold and compare models on `validation`;
`test` is evaluated only when `EVALUATE_TEST = True` – run that once, after the model is frozen.

| Output | Content |
|---|---|
| `app/model/model.json` | model card: coefficients, feature definitions, threshold, metrics, calibration, segments, input ranges |
| `app/model/model-data.js` | the same object as `window.DEADLIFT_MODEL`, so `index.html` also works when opened from disk |
| `app/tests/reference_cases.json` | inputs with Python probabilities; the JS tests must reproduce them |

## 0. Setup

In [1]:
import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss, precision_recall_curve, roc_auc_score

pd.set_option("display.width", 160)


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "docs" / "specs" / "dataset_schema.json").exists():
            return candidate
    raise FileNotFoundError("Start Jupyter from inside the repository.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATA = PROJECT_ROOT / "data" / "processed" / "deadlift3_attempts.parquet"
SCHEMA = json.loads((PROJECT_ROOT / "docs" / "specs" / "dataset_schema.json").read_text(encoding="utf-8"))
METADATA = json.loads((PROJECT_ROOT / "data" / "processed" / "metadata.json").read_text(encoding="utf-8"))
APP_MODEL_DIR = PROJECT_ROOT / "app" / "model"
APP_TEST_DIR = PROJECT_ROOT / "app" / "tests"
if not DATA.exists():
    raise FileNotFoundError("Run notebooks/01_data_extraction_and_cleaning.ipynb first (refresh-dataset skill).")
assert SCHEMA["schema_version"].split(".")[0] == "1", "notebook written for dataset schema 1.x"
# The committed model card is the frozen model. Read it now, before section 5 overwrites it, so that the test
# evaluation (section 6) can prove it evaluates exactly that model and nothing refitted or retuned since.
MODEL_CARD_PATH = APP_MODEL_DIR / "model.json"
FROZEN_CARD = json.loads(MODEL_CARD_PATH.read_text(encoding="utf-8")) if MODEL_CARD_PATH.exists() else None

In [2]:
MODEL_VERSION = "monotone-logreg-v1"
TARGET_RECALL = 0.70        # threshold rule (report 5.1): highest precision with recall >= 0.70 on validation
UNCERTAINTY_BAND = 0.02     # |p - limit| below this is shown as "close to the limit" (spec 08)
JUMP_KINKS = [0.025, 0.05, 0.075, 0.10]  # piecewise-linear jump effect, kinks at +2.5/5/7.5/10 %
PB_GAP_CLIP = 0.4           # |log(DL2 / previous best)| is clipped here (entry errors in old results)
L2 = 1e-6                   # tiny ridge penalty on standardised coefficients, for numerical stability
EVALUATE_TEST = True       # set True ONCE, after all modelling choices are frozen
N_REFERENCE_CASES = 40
RANDOM_SEED = 7

In [3]:
df = pd.read_parquet(DATA)
train, valid, test = (df[df["split"] == s].reset_index(drop=True) for s in ("train", "validation", "test"))
print(f"train {len(train):,} | validation {len(valid):,} | test {len(test):,} (test untouched unless EVALUATE_TEST)")
EQUIPMENT_LEVELS = METADATA["category_levels"]["equipment"]
AGE_LEVELS = METADATA["category_levels"]["age_class_group"]
EQUIPMENT_BASE, AGE_BASE = "Raw", "Open"

train 616,069 | validation 131,443 | test 131,953 (test untouched unless EVALUATE_TEST)


## 1. Features

Everything the coach knows after the 2nd attempt, expressed as log-ratios so that the model compares lifters of
different size fairly. The app recomputes exactly these formulas in `app/js/model.js`.

| Feature | Formula | Meaning |
|---|---|---|
| `dl2_success` | 1 / 0 | 2nd attempt good |
| `rel_strength` | ln(DL2 / bodyweight) | relative strength on the day |
| `log_bodyweight` | ln(bodyweight) | size |
| `squat_ratio`, `bench_ratio` | ln(best squat / DL2), ln(best bench / DL2) | strength profile and fatigue (Table 1 #7, #8) |
| `has_prev_best` | 1 if a previous competition best is known | first-timer indicator |
| `pb_gap`, `pb_gap_pos` | clip(ln(DL2 / previous best), ±0.4) and its positive part | how close the 2nd attempt already is to the best |
| `equipment_*`, `age_*` | one-hot, baselines Raw and Open | Table 1 #5, #6 |
| **`jump_after_good`, `jump_after_miss`** | j = ln(planned / DL2), split by the 2nd attempt | the decision |
| **`jump_over_2.5` … `jump_over_10`** | max(0, j − ln(1 + kink)) | lets the jump effect bend upwards |
| **`pr_excess`** | max(0, ln(planned / best)) − max(0, ln(DL2 / best)) | how far the attempt goes beyond the previous best |

**Bold** features grow with the planned weight. Their coefficients are constrained to be ≥ 0, so the predicted
risk can only rise as the planned weight rises. `sex` is not a feature (spec 05).

In [4]:
DECISION_FEATURES = ["jump_after_good", "jump_after_miss", *[f"jump_over_{k * 100:g}" for k in JUMP_KINKS], "pr_excess"]


def make_features(frame: pd.DataFrame, planned_kg: np.ndarray) -> pd.DataFrame:
    """Model features for a planned 3rd-attempt weight. Mirrors features() in app/js/model.js."""
    s = frame["dl2_success"].to_numpy(float)
    dl2 = frame["dl2_weight_kg"].to_numpy(float)
    bw = frame["bodyweight_kg"].to_numpy(float)
    pb = frame["prev_best_deadlift_kg"].to_numpy(float)
    has_pb = ~np.isnan(pb)
    pb_safe = np.where(has_pb, pb, 1.0)
    jump = np.log(np.asarray(planned_kg, float) / dl2)
    gap = np.where(has_pb, np.clip(np.log(dl2 / pb_safe), -PB_GAP_CLIP, PB_GAP_CLIP), 0.0)

    X = {
        "dl2_success": s,
        "rel_strength": np.log(dl2 / bw),
        "log_bodyweight": np.log(bw),
        "squat_ratio": np.log(frame["best_squat_kg"].to_numpy(float) / dl2),
        "bench_ratio": np.log(frame["best_bench_kg"].to_numpy(float) / dl2),
        "has_prev_best": has_pb.astype(float),
        "pb_gap": gap,
        "pb_gap_pos": np.maximum(gap, 0.0),
    }
    equipment = frame["equipment"].astype(str).to_numpy()
    for level in EQUIPMENT_LEVELS:
        if level != EQUIPMENT_BASE:
            X[f"equipment_{level}"] = (equipment == level).astype(float)
    age = frame["age_class_group"].astype(str).to_numpy()
    for level in AGE_LEVELS:
        if level != AGE_BASE:
            X[f"age_{level}"] = (age == level).astype(float)
    X["jump_after_good"] = jump * s
    X["jump_after_miss"] = jump * (1 - s)
    for kink in JUMP_KINKS:
        X[f"jump_over_{kink * 100:g}"] = np.maximum(0.0, jump - np.log1p(kink))
    X["pr_excess"] = np.where(
        has_pb, np.maximum(0.0, np.log(planned_kg / pb_safe)) - np.maximum(0.0, np.log(dl2 / pb_safe)), 0.0)
    return pd.DataFrame(X)


X_train = make_features(train, train["dl3_weight_kg"].to_numpy())
X_valid = make_features(valid, valid["dl3_weight_kg"].to_numpy())
y_train, y_valid = train["dl3_failed"].to_numpy(float), valid["dl3_failed"].to_numpy(float)
FEATURES = list(X_train.columns)
assert set(DECISION_FEATURES) <= set(FEATURES) and "sex" not in FEATURES
print(len(FEATURES), "features")

23 features


## 2. Fit

Maximum-likelihood logistic regression with box constraints (`scipy` L-BFGS-B). Features are standardised for
the optimiser with **train** means and standard deviations; coefficients are converted back to the raw feature
scale for export, so the app only needs a dot product and a sigmoid. For comparison the same model is fitted
without constraints.

In [5]:
def fit_logistic(X: pd.DataFrame, y: np.ndarray, monotone: bool):
    mean, std = X.mean(axis=0), X.std(axis=0).replace(0, 1.0)
    Z = ((X - mean) / std).to_numpy()
    n, k = Z.shape

    def objective(theta):
        z = theta[0] + Z @ theta[1:]
        loss = np.mean(np.logaddexp(0.0, z) - y * z) + L2 / 2 * theta[1:] @ theta[1:]
        residual = 1.0 / (1.0 + np.exp(-z)) - y
        return loss, np.concatenate([[residual.mean()], Z.T @ residual / n + L2 * theta[1:]])

    bounds = [(None, None)] + [(0.0, None) if monotone and c in DECISION_FEATURES else (None, None) for c in X.columns]
    result = minimize(objective, np.zeros(k + 1), jac=True, method="L-BFGS-B", bounds=bounds, options={"maxiter": 5000})
    assert result.success, result.message
    coef = result.x[1:] / std.to_numpy()
    intercept = result.x[0] - float(np.sum(result.x[1:] * mean.to_numpy() / std.to_numpy()))
    return intercept, pd.Series(coef, index=X.columns)


def predict(intercept: float, coef: pd.Series, X: pd.DataFrame) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-(intercept + X[coef.index].to_numpy() @ coef.to_numpy())))


intercept, coef = fit_logistic(X_train, y_train, monotone=True)
free_intercept, free_coef = fit_logistic(X_train, y_train, monotone=False)
pd.DataFrame({"monotone": coef, "unconstrained": free_coef}).round(3)

,monotone,unconstrained
dl2_success,-0.975,-0.211
rel_strength,1.685,1.651
log_bodyweight,0.964,0.940
squat_ratio,-0.336,-0.297
bench_ratio,-1.008,-0.993
has_prev_best,0.319,0.213
pb_gap,1.607,1.050
pb_gap_pos,-1.302,-0.984
equipment_Wraps,0.164,0.158
equipment_Single-ply,0.451,0.440


The unconstrained fit gives the plain jump terms large **negative** coefficients – the selection effect.
The monotone fit sets them to zero and keeps the upward bends beyond +5 % and the previous-best term: in the
data, a bigger jump only starts to cost once it is large, or once it goes past the athlete's previous best.

## 3. Validation: threshold, metrics, baselines

In [6]:
p_valid = predict(intercept, coef, X_valid)
p_valid_free = predict(free_intercept, free_coef, X_valid)


def threshold_for_recall(y, score, target):
    precision, recall, thresholds = precision_recall_curve(y, score)
    ok = recall[:-1] >= target
    best = np.argmax(np.where(ok, precision[:-1], -1))
    return float(thresholds[best])


def binary_metrics(y, flagged):
    tp = float(np.sum(flagged & (y == 1)))
    precision = tp / max(float(np.sum(flagged)), 1.0)
    recall = tp / max(float(np.sum(y == 1)), 1.0)
    return {"precision": round(precision, 4), "recall": round(recall, 4), "flagged_share": round(float(np.mean(flagged)), 4)}


THRESHOLD = round(threshold_for_recall(y_valid, p_valid, TARGET_RECALL), 4)
validation_metrics = {
    "threshold": THRESHOLD,
    **binary_metrics(y_valid, p_valid >= THRESHOLD),
    "pr_auc": round(float(average_precision_score(y_valid, p_valid)), 4),
    "roc_auc": round(float(roc_auc_score(y_valid, p_valid)), 4),
    "brier": round(float(brier_score_loss(y_valid, p_valid)), 4),
    "log_loss": round(float(log_loss(y_valid, p_valid)), 4),
    "base_rate": round(float(y_valid.mean()), 4),
    "n": int(len(y_valid)),
}

jump_valid = valid["dl3_jump_pct"].to_numpy()
b2_cut = threshold_for_recall(y_valid, jump_valid, TARGET_RECALL)
baselines = {
    "B1 always succeeds": {"precision": None, "recall": 0.0, "pr_auc": round(float(y_valid.mean()), 4)},
    f"B2 fails if jump ≥ {b2_cut:g} %" + (" (warns on every attempt)" if b2_cut <= 0 else ""): {
        **binary_metrics(y_valid, jump_valid >= b2_cut),
                                           "pr_auc": round(float(average_precision_score(y_valid, jump_valid)), 4)},
    "B3 fails if 2nd attempt failed": {**binary_metrics(y_valid, valid["dl2_success"].to_numpy() == 0),
                                       "pr_auc": round(float(average_precision_score(y_valid, 1 - valid["dl2_success"].to_numpy())), 4)},
    "Unconstrained logistic regression": {
        **binary_metrics(y_valid, p_valid_free >= threshold_for_recall(y_valid, p_valid_free, TARGET_RECALL)),
        "pr_auc": round(float(average_precision_score(y_valid, p_valid_free)), 4)},
    "Model (monotone logistic regression)": {k: validation_metrics[k] for k in ("precision", "recall", "pr_auc", "flagged_share")},
}
print("Threshold:", THRESHOLD)
pd.DataFrame(baselines).T

Threshold: 0.3541


,precision,recall,pr_auc,flagged_share
B1 always succeeds,NaN,0.0000,0.3860,NaN
B2 fails if jump ≥ 0 % (warns on every attempt),0.3860,1.0000,0.3441,1.0000
B3 fails if 2nd attempt failed,0.6233,0.1805,0.4288,0.1118
Unconstrained logistic regression,0.4855,0.7000,0.5553,0.5566
Model (monotone logistic regression),0.4831,0.7002,0.5507,0.5594


### Does a heavier attempt ever look safer?

For 2,000 validation situations, sweep the planned weight from DL2 to +25 % and count situations in which the
predicted risk goes *down* somewhere along the way.

In [7]:
def sweep(intercept_, coef_, frame, steps=26):
    factors = np.linspace(1.0, 1.25, steps)
    return np.stack([predict(intercept_, coef_, make_features(frame, frame["dl2_weight_kg"].to_numpy() * f))
                     for f in factors], axis=1)


contexts = valid.sample(2000, random_state=RANDOM_SEED)
monotonicity = {}
for name, (b0, b) in {"monotone": (intercept, coef), "unconstrained": (free_intercept, free_coef)}.items():
    curves = sweep(b0, b, contexts)
    monotonicity[name] = {
        "situations_where_more_weight_looks_safer": round(float((np.diff(curves, axis=1) < -1e-12).any(axis=1).mean()), 4),
        "median_risk_change_plus_5pct": round(float(np.median(curves[:, 5] - curves[:, 0])), 4),
        "median_risk_change_plus_10pct": round(float(np.median(curves[:, 10] - curves[:, 0])), 4),
    }
assert monotonicity["monotone"]["situations_where_more_weight_looks_safer"] == 0
pd.DataFrame(monotonicity).T

,situations_where_more_weight_looks_safer,median_risk_change_plus_5pct,median_risk_change_plus_10pct
monotone,0.0,0.0015,0.0821
unconstrained,1.0,-0.2164,-0.0882


### Calibration (validation)

A predicted 40 % should fail about 40 % of the time – the app shows probabilities, so calibration matters.

In [8]:
bins = pd.qcut(p_valid, 10, duplicates="drop")
calibration = (pd.DataFrame({"p": p_valid, "y": y_valid, "bin": bins})
               .groupby("bin", observed=True).agg(predicted=("p", "mean"), observed=("y", "mean"), n=("y", "size")))
calibration = calibration.round(4).reset_index(drop=True)
calibration

,predicted,observed,n
0,0.1739,0.1801,13145
1,0.2455,0.2390,13144
2,0.2883,0.2818,13144
3,0.3239,0.3154,13144
4,0.3573,0.3590,13145
5,0.3906,0.3840,13144
6,0.4268,0.4342,13145
7,0.4687,0.4701,13143
8,0.5259,0.5351,13144
9,0.6635,0.6612,13145


### Segments (validation, report 6.2)

Precision and recall at the global threshold. `sex` is used **only here**. Segments with fewer than 100 failed
attempts are marked unreliable. These numbers come from the validation split; the final Table 4 of the report
uses test once the model is frozen.

In [9]:
def segment_table(frame, p, y, threshold):
    rows = []
    for column in ("sex", "age_class_group", "equipment"):
        for value, idx in frame.groupby(column, observed=True).indices.items():
            m = binary_metrics(y[idx], p[idx] >= threshold)
            positives = int(y[idx].sum())
            rows.append({"segment": column, "value": str(value), "n": int(len(idx)), "failed": positives,
                         "precision": m["precision"], "recall": m["recall"], "reliable": positives >= 100})
    table = pd.DataFrame(rows)
    overall = binary_metrics(y, p >= threshold)["recall"]
    table["recall_gap_pp"] = ((table["recall"] - overall) * 100).round(1)
    table["flag"] = np.where(~table["reliable"], "too small", np.where(table["recall_gap_pp"].abs() > 10, "check", ""))
    return table


segments = segment_table(valid, p_valid, y_valid, THRESHOLD)
segments

,segment,value,n,failed,precision,recall,reliable,recall_gap_pp,flag
0,sex,M,87031,35714,0.4846,0.7897,True,8.9,
1,sex,F,44396,15017,0.4777,0.4874,True,-21.3,check
2,sex,Mx,16,3,0.0000,0.0000,False,-70.0,too small
3,age_class_group,Sub-Junior,16791,5918,0.4745,0.5906,True,-11.0,check
4,age_class_group,Junior,32806,13085,0.4852,0.7391,True,3.9,
5,age_class_group,Open,52693,21207,0.4891,0.7424,True,4.2,
6,age_class_group,Masters,20354,7157,0.4649,0.6120,True,-8.8,
7,age_class_group,Unknown,8799,3367,0.4836,0.6629,True,-3.7,
8,equipment,Raw,105004,39347,0.4810,0.6703,True,-3.0,
9,equipment,Wraps,16757,7008,0.4876,0.7770,True,7.7,


## 4. Typical ranges for the app

The app accepts any value inside the schema bounds but flags inputs outside the 1st–99th percentile of the
**training** data as unusual, both for raw inputs and for the athlete profile (the model features). Outside those
ranges the model extrapolates.

In [10]:
schema_cols = {c["name"]: c for c in SCHEMA["columns"]}
INPUTS = ["dl2_weight_kg", "bodyweight_kg", "best_squat_kg", "best_bench_kg", "prev_best_deadlift_kg"]
input_bounds = {name: {"min": schema_cols[name]["min"], "max": schema_cols[name]["max"]} for name in INPUTS}
input_typical = {name: {"p01": round(METADATA["feature_ranges_train"][name]["p01"], 2),
                        "p99": round(METADATA["feature_ranges_train"][name]["p99"], 2)} for name in INPUTS}
PROFILE = ["rel_strength", "squat_ratio", "bench_ratio", "pb_gap"]
profile_typical = {}
for name in PROFILE:
    values = X_train[name] if name != "pb_gap" else X_train.loc[X_train["has_prev_best"] == 1, name]
    profile_typical[name] = {"p01": round(float(values.quantile(0.01)), 4), "p99": round(float(values.quantile(0.99)), 4)}
pd.DataFrame({**input_typical, **profile_typical}).T

,p01,p99
dl2_weight_kg,75.0000,325.0000
bodyweight_kg,46.6300,146.5500
best_squat_kg,57.5000,335.0000
best_bench_kg,35.0000,225.0000
prev_best_deadlift_kg,83.9100,340.0000
rel_strength,0.0656,1.3063
squat_ratio,-0.4912,0.2144
bench_ratio,-1.0647,-0.1198
pb_gap,-0.4000,0.2559


## 5. Export

In [11]:
def feature_doc(name):
    docs = {
        "dl2_success": "1 if the 2nd attempt was good", "rel_strength": "ln(DL2 / bodyweight)",
        "log_bodyweight": "ln(bodyweight)", "squat_ratio": "ln(best squat / DL2)", "bench_ratio": "ln(best bench / DL2)",
        "has_prev_best": "1 if a previous competition best is given",
        "pb_gap": f"clip(ln(DL2 / previous best), -{PB_GAP_CLIP}, {PB_GAP_CLIP}), 0 without previous best",
        "pb_gap_pos": "max(pb_gap, 0)", "jump_after_good": "ln(planned / DL2) if the 2nd attempt was good",
        "jump_after_miss": "ln(planned / DL2) if the 2nd attempt was missed",
        "pr_excess": "max(0, ln(planned / previous best)) - max(0, ln(DL2 / previous best))",
    }
    if name.startswith("jump_over_"):
        return f"max(0, ln(planned / DL2) - ln(1 + {name.split('_')[-1]} %))"
    if name.startswith("equipment_"):
        return f"1 if equipment is {name.removeprefix('equipment_')} (baseline {EQUIPMENT_BASE})"
    if name.startswith("age_"):
        return f"1 if age group is {name.removeprefix('age_')} (baseline {AGE_BASE})"
    return docs[name]



def is_frozen_model(card) -> bool:
    """True if `card` holds the model fitted in this run: same data snapshot, threshold and predictions.

    Predictions are compared instead of raw coefficients, so tiny optimiser differences between machines
    (< 1e-6 in probability) do not count as a different model.
    """
    if not card or card.get("model_version") != MODEL_VERSION:
        return False
    if card["dataset"].get("opl_commit") != METADATA["source"]["opl_commit"] or card["threshold"] != THRESHOLD:
        return False
    if set(card["coefficients"]) != set(coef.index):
        return False
    p_card = predict(card["intercept"], pd.Series(card["coefficients"])[coef.index], X_valid)
    return float(np.max(np.abs(p_card - p_valid))) < 1e-6


model_card = {
    "model_version": MODEL_VERSION,
    "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "model_type": "Logistic regression, monotone (non-decreasing) in the planned 3rd-attempt weight",
    "target": "probability that the 3rd deadlift attempt fails (dl3_failed = 1)",
    "dataset": {
        "snapshot_date": METADATA["source"]["snapshot_date"], "opl_commit": METADATA["source"]["opl_commit"],
        "schema_version": SCHEMA["schema_version"], "period": METADATA["summary"]["period"],
        "train_records": int(len(train)), "train_athletes": int(train["athlete_id"].nunique()),
        "validation_records": int(len(valid)),
    },
    "features": [{"name": n, "formula": feature_doc(n), "grows_with_planned_weight": n in DECISION_FEATURES} for n in FEATURES],
    "intercept": float(intercept),                              # full precision: the app must match exactly
    "coefficients": {n: float(v) for n, v in coef.items()},
    "constants": {"jump_kinks": JUMP_KINKS, "pb_gap_clip": PB_GAP_CLIP},
    "categories": {"equipment": EQUIPMENT_LEVELS, "equipment_baseline": EQUIPMENT_BASE,
                   "age_class_group": AGE_LEVELS, "age_baseline": AGE_BASE},
    "threshold": THRESHOLD,
    "threshold_rule": f"highest precision with recall >= {TARGET_RECALL} on the validation split",
    "uncertainty_band": UNCERTAINTY_BAND,
    "max_jump_pct": METADATA["parameters"]["max_jump_pct"],
    "input_bounds": input_bounds, "input_typical": input_typical, "profile_typical": profile_typical,
    "metrics": {"validation": validation_metrics, "test": None},
    "baselines_validation": baselines,
    "monotonicity_check_validation": monotonicity,
    "calibration_validation": calibration.to_dict(orient="records"),
    "segments_validation": segments.to_dict(orient="records"),
    "limitations": [
        "Learned from competition results: coaches already pick smaller jumps after a slow 2nd attempt, so the data "
        "cannot show how much safer a small jump is for a given athlete. Up to about +5 % the predicted risk is flat.",
        "Day form, warm-ups, bar speed and weight cutting are not in the data. Treat the output as one input to the "
        "coach's decision.",
        "Federations judge differently, and only the last declared 3rd-attempt weight is recorded.",
        "Small groups (Mx, Multi-ply, Unlimited, very young or old lifters) have few cases; predictions there are "
        "less reliable.",
    ],
    "attribution": METADATA["source"]["attribution"],
}
TEST_KEYS = ("baselines_test", "segments_test")
if FROZEN_CARD and FROZEN_CARD["metrics"].get("test"):
    if is_frozen_model(FROZEN_CARD):
        # Same frozen model: keep its one-time test results (and its creation date) instead of dropping them.
        model_card["created_utc"] = FROZEN_CARD["created_utc"]
        model_card["metrics"]["test"] = FROZEN_CARD["metrics"]["test"]
        model_card.update({k: FROZEN_CARD[k] for k in TEST_KEYS if k in FROZEN_CARD})
        print("Frozen model unchanged – kept its test results from", FROZEN_CARD["metrics"]["test"]["evaluated_utc"])
    else:
        print("WARNING: the model differs from the one evaluated on test on",
              FROZEN_CARD["metrics"]["test"]["evaluated_utc"], "– its test results were dropped. The test set has "
              "been used before: say so in the report (spec 06, rule 4).")
APP_MODEL_DIR.mkdir(parents=True, exist_ok=True)
APP_TEST_DIR.mkdir(parents=True, exist_ok=True)
payload = json.dumps(model_card, indent=2, ensure_ascii=False, default=lambda v: v.item() if hasattr(v, "item") else str(v))
(APP_MODEL_DIR / "model.json").write_text(payload + "\n", encoding="utf-8")
(APP_MODEL_DIR / "model-data.js").write_text(
    "// Generated by notebooks/02_model_training.ipynb – do not edit by hand.\n"
    f"window.DEADLIFT_MODEL = {payload};\n", encoding="utf-8")
print("wrote", (APP_MODEL_DIR / "model.json").relative_to(PROJECT_ROOT), "and model-data.js")

wrote app/model/model.json and model-data.js


In [12]:
# Reference cases: validation situations with candidate weights and the Python probability.
# app/tests/model.test.js checks that the JavaScript implementation reproduces them.
cases = valid.sample(N_REFERENCE_CASES, random_state=RANDOM_SEED).reset_index(drop=True)
factor = np.random.default_rng(RANDOM_SEED).uniform(1.0, 1.2, len(cases))
dl2_cases = cases["dl2_weight_kg"].to_numpy()
planned = np.maximum(np.round(dl2_cases * factor, 1), dl2_cases)
planned = np.where(cases["dl2_success"].to_numpy() == 1, np.maximum(planned, dl2_cases + 0.5), planned)
p_cases = predict(intercept, coef, make_features(cases, planned))
reference = []
for i, row in cases.iterrows():
    reference.append({
        "context": {
            "dl2_weight_kg": float(row.dl2_weight_kg), "dl2_success": bool(row.dl2_success),
            "bodyweight_kg": float(row.bodyweight_kg), "age_class_group": str(row.age_class_group),
            "equipment": str(row.equipment), "best_squat_kg": float(row.best_squat_kg),
            "best_bench_kg": float(row.best_bench_kg),
            "prev_best_deadlift_kg": None if pd.isna(row.prev_best_deadlift_kg) else float(row.prev_best_deadlift_kg),
        },
        "planned_kg": float(planned[i]),
        "p_fail": float(p_cases[i]),
    })
(APP_TEST_DIR / "reference_cases.json").write_text(json.dumps({"model_version": MODEL_VERSION, "cases": reference}, indent=1) + "\n")
print(len(reference), "reference cases written")

40 reference cases written


## 6. Final evaluation on test (run once)

The test split (15 % of the athlete ids) is touched **once**, with the frozen model:

* **Frozen model** = the committed `app/model/model.json` (logistic regression, monotone in the planned weight,
  fitted on train) and its threshold chosen on validation (0.3541, rounded 0.35 in the report).
* The cell refuses to run if this run's model differs from the committed one (other data snapshot, changed
  features, other threshold). Nothing is refitted or tuned on test; the split is the exported `split` column.
* It computes the numbers for the report in one pass: precision, recall and PR-AUC of the model (Table 3,
  Executive Summary), the baselines B1–B3 with their validation cut-offs (Table 3) and the segment table (Table 4).
* The result and the date are stored in the model card (`metrics.test`, `baselines_test`, `segments_test`);
  later re-runs of the same frozen model keep them unchanged.

**How to run:** set `EVALUATE_TEST = True` in section 0, *Run All*, commit the notebook and `app/model/*`, and
add the date to `docs/decisions.md`. If the test numbers are worse than validation, report them as they are –
changing the model or threshold afterwards would turn the test set into a second validation set.

In [13]:
if EVALUATE_TEST:
    from sklearn.metrics import precision_score, recall_score

    if not is_frozen_model(FROZEN_CARD):
        raise RuntimeError(
            "This run's model is not the committed app/model/model.json (the frozen model): different data snapshot, "
            "features, coefficients or threshold. Restore the committed state, or commit the model you want to freeze "
            "first – and remember that the test set may be used only once.")
    previous = FROZEN_CARD["metrics"].get("test")
    if previous:
        print("Test already evaluated on", previous["evaluated_utc"], "– reproducing the same numbers for the same frozen model.")

    frozen_intercept = FROZEN_CARD["intercept"]
    frozen_coef = pd.Series(FROZEN_CARD["coefficients"])[coef.index]
    X_test = make_features(test, test["dl3_weight_kg"].to_numpy())
    y_test = test["dl3_failed"].to_numpy(float)
    p_test = predict(frozen_intercept, frozen_coef, X_test)
    flagged = p_test >= THRESHOLD
    evaluated = previous["evaluated_utc"] if previous else datetime.now(timezone.utc).isoformat(timespec="seconds")

    test_metrics = {
        "threshold": THRESHOLD,
        **binary_metrics(y_test, flagged),
        "pr_auc": round(float(average_precision_score(y_test, p_test)), 4),
        "roc_auc": round(float(roc_auc_score(y_test, p_test)), 4),
        "brier": round(float(brier_score_loss(y_test, p_test)), 4),
        "base_rate": round(float(y_test.mean()), 4),
        "n": int(len(y_test)),
        "evaluated_utc": evaluated,
    }
    # Cross-check against scikit-learn's definitions
    assert abs(precision_score(y_test, flagged) - test_metrics["precision"]) < 1e-4
    assert abs(recall_score(y_test, flagged) - test_metrics["recall"]) < 1e-4

    # Baselines with the cut-offs fixed on validation (section 3) – nothing is chosen on test
    jump_test, dl2_test = test["dl3_jump_pct"].to_numpy(), test["dl2_success"].to_numpy()
    b2_name = next(name for name in baselines if name.startswith("B2"))
    baselines_test = {
        "B1 always succeeds": {"precision": None, "recall": 0.0, "pr_auc": round(float(y_test.mean()), 4)},
        b2_name: {**binary_metrics(y_test, jump_test >= b2_cut),
                  "pr_auc": round(float(average_precision_score(y_test, jump_test)), 4)},
        "B3 fails if 2nd attempt failed": {**binary_metrics(y_test, dl2_test == 0),
                                           "pr_auc": round(float(average_precision_score(y_test, 1 - dl2_test)), 4)},
        "Model (monotone logistic regression)": {k: test_metrics[k] for k in ("precision", "recall", "pr_auc", "flagged_share")},
    }
    segments_test = segment_table(test, p_test, y_test, THRESHOLD)

    model_card["metrics"]["test"] = test_metrics
    model_card["baselines_test"] = baselines_test
    model_card["segments_test"] = segments_test.to_dict(orient="records")
    payload = json.dumps(model_card, indent=2, ensure_ascii=False, default=lambda v: v.item() if hasattr(v, "item") else str(v))
    (APP_MODEL_DIR / "model.json").write_text(payload + "\n", encoding="utf-8")
    (APP_MODEL_DIR / "model-data.js").write_text(
        "// Generated by notebooks/02_model_training.ipynb – do not edit by hand.\n"
        f"window.DEADLIFT_MODEL = {payload};\n", encoding="utf-8")

    day = datetime.fromisoformat(evaluated).strftime("%d.%m.%Y")
    print(f"Für den Bericht – Testset ({test_metrics['n']:,} Versuche), ausgewertet am {day}, Schwellenwert {THRESHOLD}:")
    print(f"  Precision «scheitert» {test_metrics['precision']:.2f} · Recall «scheitert» {test_metrics['recall']:.2f} · "
          f"PR-AUC {test_metrics['pr_auc']:.2f}  (Validierung: {validation_metrics['precision']:.2f} · "
          f"{validation_metrics['recall']:.2f} · {validation_metrics['pr_auc']:.2f})")
    display(pd.DataFrame(baselines_test).T.rename_axis("Tabelle 3 (Test)"))
    display(segments_test.rename_axis("Tabelle 4 (Test)"))
else:
    print("Test set not evaluated (EVALUATE_TEST = False). The app shows validation metrics.")

Für den Bericht – Testset (131,953 Versuche), ausgewertet am 10.10.2026, Schwellenwert 0.3541:
  Precision «scheitert» 0.48 · Recall «scheitert» 0.70 · PR-AUC 0.55  (Validierung: 0.48 · 0.70 · 0.55)


,precision,recall,pr_auc,flagged_share
Tabelle 3 (Test),,,,
B1 always succeeds,NaN,0.0000,0.3872,NaN
B2 fails if jump ≥ 0 % (warns on every attempt),0.3872,1.0000,0.3486,1.0000
B3 fails if 2nd attempt failed,0.6066,0.1764,0.4259,0.1126
Model (monotone logistic regression),0.4817,0.7022,0.5490,0.5644


,segment,value,n,failed,precision,recall,reliable,recall_gap_pp,flag
Tabelle 4 (Test),,,,,,,,,
0,sex,M,87952,36238,0.4834,0.7894,True,8.7,
1,sex,F,43965,14848,0.4750,0.4893,True,-21.3,check
2,sex,Mx,36,6,0.5000,0.3333,False,-36.9,too small
3,age_class_group,Sub-Junior,16711,5893,0.4726,0.6026,True,-10.0,
4,age_class_group,Junior,32623,13259,0.4887,0.7405,True,3.8,
5,age_class_group,Open,52472,20975,0.4859,0.7452,True,4.3,
6,age_class_group,Masters,21158,7591,0.4646,0.6144,True,-8.8,
7,age_class_group,Unknown,8989,3374,0.4740,0.6556,True,-4.7,
8,equipment,Raw,105064,39272,0.4745,0.6669,True,-3.5,
